In [2]:
from sklearn.metrics.pairwise import euclidean_distances, cosine_distances
from pathlib import Path
import pandas as pd
import os

In [ ]:
path_work_dir = Path("/home/b05b01002/HDD/project_scRNAed")
path_output = path_work_dir / "outputs/notebooks/Show-DRE-sites-on-Seurat-UMAP/"
path_embeddings = path_work_dir / "outputs/notebooks/Show-Proteome-Validated-on-Seurat-UMAP/integrated.cca.csv"
path_cell_ident = path_work_dir / "references/ptr_tenx_batch1_rs17_curated.csv"
os.mkdir(path_output)

### Get cell identity by nearest reference

Read reference cell identity

In [34]:
reference_cell_ident = pd.read_csv(path_cell_ident, index_col=0)
reference_cell_ident.head()

,Cluster,Color
Barcode,,
AAACCTGAGCACCGCT-1,8,#E377C2
AAACCTGCAAGAGTCG-1,7,#D62728
AAACCTGCACATTCGA-1,3,#FF7F0F
AAACCTGGTAAATGAC-1,3,#FF7F0F
AAACCTGGTCATCGGC-1,1,#1F77B4


Read embeddings of all bioreps

In [11]:
embeddings = pd.read_csv(
    path_embeddings,
    index_col=0
)
embeddings.head()

,integratedcca_1,integratedcca_2,integratedcca_3,integratedcca_4,integratedcca_5,integratedcca_6,integratedcca_7,integratedcca_8,integratedcca_9,integratedcca_10,...,integratedcca_21,integratedcca_22,integratedcca_23,integratedcca_24,integratedcca_25,integratedcca_26,integratedcca_27,integratedcca_28,integratedcca_29,integratedcca_30
AAACCTGAGCACCGCT-1_1,8.553631,-6.008619,-10.468839,-5.856133,-48.376924,45.620813,2.572424,19.380106,6.624714,-20.053965,...,0.405844,0.531237,-8.313060,0.689239,3.406736,-3.707216,-0.181881,0.115383,1.401048,1.605206
AAACCTGCAAGAGTCG-1_1,-117.451413,-9.682598,-3.159338,19.702042,2.412085,3.317594,12.266135,2.274045,0.714785,2.188559,...,-0.733071,-0.135228,6.041794,4.580537,12.267207,-3.583874,-4.448090,-1.436238,0.661395,1.226619
AAACCTGCACATTCGA-1_1,15.869051,-39.146500,-2.289393,-1.813494,6.553414,8.280496,0.225023,-24.714781,3.586793,-2.418766,...,1.516785,5.197331,-3.914603,8.858530,0.322296,0.372711,4.294704,4.169678,2.518136,3.936872
AAACCTGGTAAATGAC-1_1,10.859940,-20.811009,-26.278891,0.547737,3.571729,5.792480,-2.214079,-10.171792,0.004572,1.603109,...,0.216264,1.029186,4.033681,-12.215694,1.839143,-2.549375,-1.396900,1.570853,-1.299284,-3.293464
AAACCTGGTCATCGGC-1_1,-0.468537,17.389900,7.851869,0.961913,0.063824,-3.162259,1.174197,-2.405853,-0.634247,-0.679696,...,0.973016,4.740723,-1.487619,-1.427436,0.847993,-3.265063,-0.791963,-3.914409,-3.453628,-3.225413


Calculate pairwise distances

In [50]:
distance = euclidean_distances(embeddings, embeddings.loc[["%s_1" % (i) for i in  reference_cell_ident.index]])

In [51]:
distance = pd.DataFrame(
    distance,
    index=embeddings.index,
    columns=reference_cell_ident.index
)

In [52]:
nearest_reference = distance.idxmin(axis=1)

In [59]:
cell_identity = pd.DataFrame(
    dict(
        Barcode=nearest_reference.index,
        Cluster=reference_cell_ident.loc[nearest_reference, "Cluster"].values,
    )
)
cell_identity.to_csv(
    path_output / "cell_identity_all.csv",
    index=False
)

### Plot RNA editing level on UMAP

In [4]:
from scipy.io import mmread

sample_names = [
    "ptr_tenx_batch1",
    "ptr_tenx_batch2",
    "ptr_tenx_tsv1",
    "ptr_tenx_tsv2",
    "ptr_tenx_tsv3",
    "ptr_tenx_tsv4",
    "ptr_tenx_tsv5"
]
path_barcode = path_work_dir / "outputs/Remapping/renamer/{sample}/barcodes.tsv"
path_snv_loci = path_work_dir / "outputs/VariantCalling/vawk/{sample}.snv.loci.txt"
path_alt_mtx = path_work_dir / "outputs/VariantCalling/vartrix/{sample}/alt.mtx"
path_ref_mtx = path_work_dir / "outputs/VariantCalling/vartrix/{sample}/ref.mtx"
path_DRE_RO_vs_FuO = path_work_dir / "outputs/notebooks/Differential-analysis/DRES_RO_vs_FuO.csv"
path_DRE_RO_vs_Others = path_work_dir / "outputs/notebooks/Differential-analysis/DRES_RO_vs_Others.csv"
path_variant_annotation = path_work_dir / "outputs/notebooks/Check-proteome-alignment-results/variant_annotation_6frames.tsv"
path_annotation_info = path_work_dir / "references/Ptr/annotation/Ptrichocarpa_533_v4.1.annotation_info.txt"

# read data
DRE_RO_vs_FuO = pd.read_csv(path_DRE_RO_vs_FuO, sep=",")
DRE_RO_vs_Others = pd.read_csv(path_DRE_RO_vs_Others, sep=",")
variant_annotation = pd.read_csv(path_variant_annotation, sep="\t")
annotation_info = pd.read_csv(path_annotation_info, sep="\t")

# set index to dataframes
variant_annotation.index = variant_annotation.loc[:, ["CHROM", "POS"]].apply(lambda x: f"{x.CHROM}:{x.POS}", 1)
annotation_info.index = annotation_info["locusName"]

In [5]:
loci_of_interest = set(DRE_RO_vs_FuO["locus"].values) | set(DRE_RO_vs_Others["locus"].values)
loci_of_interest_annotated = set(variant_annotation.index) & loci_of_interest
loci_of_interest_annotated = list(loci_of_interest_annotated)

In [ ]:
gene_of_interest = variant_annotation.loc[loci_of_interest_annotated,"GENEID"].apply(lambda x: "%s.%s" % (x.split(".")[0], x.split(".")[1])).unique()
annotation_info_of_interest = annotation_info.loc[gene_of_interest]
annotation_info_of_interest.to_csv(
    path_output / "annotation_info_DRE_genes.csv",
    index=False
)

locusName
Potri.002G072100    AT1G77120
Potri.009G085100    AT1G50010
Potri.017G113300    AT5G16050
Potri.017G113300    AT5G16050
Potri.017G113300    AT5G16050
Potri.018G126400    AT5G20250
Potri.018G126400    AT5G20250
Potri.018G126400    AT5G20250
Potri.008G069400    AT3G55980
Potri.014G071200    AT2G45450
Potri.011G119700    AT4G27500
Potri.011G119700    AT4G27500
Potri.011G119700    AT4G27500
Potri.010G208300    AT5G54980
Potri.012G133950          NaN
Potri.012G113500    AT5G07440
Potri.005G219700    AT1G04270
Potri.007G099400          NaN
Potri.002G140100    AT3G10950
Potri.006G136700    AT5G20830
Potri.006G136700    AT5G20830
Potri.006G136700    AT5G20830
Potri.006G136700    AT5G20830
Potri.006G136700    AT5G20830
Potri.006G136700    AT5G20830
Potri.006G136700    AT5G20830
Potri.008G051200    AT2G28710
Potri.002G106800    AT1G09970
Potri.012G077300    AT5G50090
Potri.001G069100    AT3G61110
Potri.002G191400    AT2G47170
Potri.007G086000    AT5G65207
Potri.013G054200    AT3G04060


In [12]:
variant_annotation.loc[loci_of_interest_annotated].to_csv(
    path_output / "variant_annotation_DRE_genes.csv",
    index=False
)

In [98]:
variant_annotation["GENE_NAME"] = variant_annotation.apply(lambda x: "Potri.%s.v4.1" % (x.GENEID.split(".")[1]), 1)

In [99]:
loci_to_gene = variant_annotation.loc[loci_of_interest_annotated,:].apply(lambda x: f"{x.CHROM}-{x.POS}_{x.GENE_NAME}", 1)
loci_to_gene = {idx: value for idx, value in zip(loci_to_gene.index, loci_to_gene)}
loci_to_gene

{'Chr02:5058364': 'Chr02-5058364_Potri.002G072100.v4.1',
 'Chr09:7998571': 'Chr09-7998571_Potri.009G085100.v4.1',
 'Chr17:12070245': 'Chr17-12070245_Potri.017G113300.v4.1',
 'Chr18:13765592': 'Chr18-13765592_Potri.018G126400.v4.1',
 'Chr02:5058404': 'Chr02-5058404_Potri.002G072100.v4.1',
 'Chr08:4239255': 'Chr08-4239255_Potri.008G069400.v4.1',
 'Chr14:4538500': 'Chr14-4538500_Potri.014G071200.v4.1',
 'Chr11:15074200': 'Chr11-15074200_Potri.011G119700.v4.1',
 'Chr10:19818286': 'Chr10-19818286_Potri.010G208300.v4.1',
 'Chr12:14922345': 'Chr12-14922345_Potri.012G133950.v4.1',
 'Chr12:13302741': 'Chr12-13302741_Potri.012G113500.v4.1',
 'Chr05:22214891': 'Chr05-22214891_Potri.005G219700.v4.1',
 'Chr07:12410852': 'Chr07-12410852_Potri.007G099400.v4.1',
 'Chr02:10512592': 'Chr02-10512592_Potri.002G140100.v4.1',
 'Chr06:11264784': 'Chr06-11264784_Potri.006G136700.v4.1',
 'Chr08:3004928': 'Chr08-3004928_Potri.008G051200.v4.1',
 'Chr02:7891079': 'Chr02-7891079_Potri.002G106800.v4.1',
 'Chr12:100

In [104]:
alt_frac_all = None
for idx, i in enumerate(sample_names):
    loci = pd.read_csv(path_snv_loci.__str__().replace("{sample}", i), header=None)
    is_loci_of_interest = loci[0].isin(loci_of_interest)
    print("Found", sum(is_loci_of_interest), "loci in sample:", i)
    barcode = pd.read_csv(path_barcode.__str__().replace("{sample}", i), header=None)
    alt_mtx_temp = mmread(path_alt_mtx.__str__().replace("{sample}", i)).tocsr()
    ref_mtx_temp = mmread(path_ref_mtx.__str__().replace("{sample}", i)).tocsr()
    alt_mtx_temp = alt_mtx_temp[is_loci_of_interest, ]
    ref_mtx_temp = ref_mtx_temp[is_loci_of_interest, ]
    cov_mtx_temp = alt_mtx_temp + ref_mtx_temp
    alt_frac_temp = alt_mtx_temp / cov_mtx_temp
    alt_frac = pd.DataFrame(alt_frac_temp.T)
    alt_frac.columns = loci.loc[is_loci_of_interest, 0]
    alt_frac.index = [f"{i}_{idx+1}" for i in barcode[0]]
    alt_frac[pd.DataFrame.sparse.from_spmatrix(cov_mtx_temp) < 5] = 0
    alt_frac[alt_frac.isna()] = 0
    if alt_frac_all is None:
        alt_frac_all = alt_frac
    else:
        alt_frac_all = pd.concat([alt_frac_all, alt_frac], sort=False)

alt_frac_all[alt_frac_all.isna()] = 0

Found 33 loci in sample: ptr_tenx_batch1
Found 5 loci in sample: ptr_tenx_batch2
Found 11 loci in sample: ptr_tenx_tsv1
Found 6 loci in sample: ptr_tenx_tsv2
Found 3 loci in sample: ptr_tenx_tsv3
Found 2 loci in sample: ptr_tenx_tsv4
Found 3 loci in sample: ptr_tenx_tsv5


In [105]:
alt_frac_all = alt_frac_all.rename(
    loci_to_gene, axis = 1
)

In [106]:
alt_frac_all.to_csv(
    path_output / "editing_level.csv"
)